# L21 demo: an eval harness, a judge you validate, and a surrogate you audit by slice

Two parallel eval exercises, matching the module's two tracks. **Part A** builds a
~60-line eval harness for a small RAG assistant (in the spirit of L17's system): a
frozen eval set, two programmatic checks, an LLM-as-judge stand-in, and a real
human-agreement check against that judge, logged to MLflow. **Part B** evaluates a
regression model on real L3/L4 Intel Lab sensor data with a parity check, per-slice
MAE, a bootstrap confidence interval, and a calibration check, exactly the "one
number hides the failure that matters" argument from the notes, with real numbers.

**One honest substitution.** There is no hosted LLM in this sandbox (no API key, no
network path to a provider), so the "LLM-as-judge" below is a small, explicit,
deliberately naive heuristic: it scores an answer by lexical overlap with the
reference text alone. It shows what a judge validation exercise looks like and why
you run one. How well a real judge would perform is a separate question this
heuristic cannot answer. A real LLM judge would likely do much better on some of the
cases below and would plausibly share others' blind spots. The methodology, write a
rubric, hand-label a sample, measure agreement, distrust a low kappa, is identical
either way.

## Part A: an eval harness for a RAG assistant

### The corpus and the frozen eval set

A small constructed engineering-manual corpus (a subset of L17's), and a versioned
eval set: question, reference answer, and which chunk id the answer is required to
cite. This is the artifact CLAUDE.md's own conventions would have you commit to the
repo: change the corpus or the pipeline all you like, this file's questions and
reference answers do not move, so every run is scored against the same target.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

DOCS = {
    'A': {'title': 'Copper Tube Field Bending Practice Guide', 'sections': [
        ('4.1', 'General bend radius rule',
         "For soft-annealed copper tube formed by hand without a bending tool, the "
         "minimum bend radius is four times the tube's outer diameter."),
        ('4.2', 'Minimum radius for 12 mm tube',
         "For 12 mm outer diameter soft copper tube, the minimum bend radius using "
         "a manual lever-type bender is 48 mm."),
    ]},
    'C': {'title': 'Pressure Vessel Inspection Intervals', 'sections': [
        ('2.1', 'Internal inspection interval',
         "Vessels in continuous service must receive an internal visual inspection "
         "no less often than every five years."),
        ('2.2', 'Hydrostatic test pressure',
         "Where a hydrostatic retest is required, the test pressure is 1.5 times "
         "the vessel's maximum allowable working pressure."),
    ]},
    'E': {'title': 'Fastener Torque Specifications', 'sections': [
        ('5.1', 'Grade 5 bolt torque table',
         "Dry, unlubricated Grade 5 bolt torque values: 1/4 inch, 8 ft-lb; "
         "3/8 inch, 30 ft-lb; 1/2 inch, 72 ft-lb."),
    ]},
}

CHUNKS = [
    {'id': f'{doc_id}-{sec}', 'citation': f'[{doc_id} \u00a7{sec}]',
     'text': f"{doc['title']}, {sec} {heading}: {text}"}
    for doc_id, doc in DOCS.items() for sec, heading, text in doc['sections']
]

EVAL_SET = [
    {'question': 'What is the minimum bend radius for 12 mm copper tube?',
     'reference_answer': '48 mm', 'must_cite': 'A-4.2'},
    {'question': 'How often must a pressure vessel receive an internal inspection?',
     'reference_answer': 'every five years', 'must_cite': 'C-2.1'},
    {'question': 'What hydrostatic test pressure is required for a vessel retest?',
     'reference_answer': '1.5 times the maximum allowable working pressure', 'must_cite': 'C-2.2'},
    {'question': 'What torque should be applied to a 3/8 inch Grade 5 bolt?',
     'reference_answer': '30 ft-lb', 'must_cite': 'E-5.1'},
]
print(len(CHUNKS), 'chunks;', len(EVAL_SET), 'frozen eval questions')


### The system under test, and its trace

A minimal retriever (TF-IDF + truncated SVD, as in L17), and a deliberately simple
extractive "answer": return the top retrieved chunk's text, cited by chunk id. Every
call records a **trace**, exactly the observability topic from the notes: the
retrieved candidates, the chunk actually cited, and the wall-clock latency. This is
a JSON-serializable dict on purpose, the same shape a structured log line would
take in a real system.

In [ ]:
import time

texts = [c['text'] for c in CHUNKS]
tfidf = TfidfVectorizer(stop_words='english').fit(texts)
X = tfidf.transform(texts)
svd = TruncatedSVD(n_components=min(8, X.shape[1] - 1), random_state=0).fit(X)
emb = normalize(svd.transform(X))

def retrieve(query, k=2):
    qv = normalize(svd.transform(tfidf.transform([query])))
    scores = emb @ qv[0]
    order = np.argsort(scores)[::-1][:k]
    return [(CHUNKS[i], float(scores[i])) for i in order]

def run_system(question, inject_hallucinated_citation=False):
    t0 = time.perf_counter()
    retrieved = retrieve(question, k=2)
    top_chunk, top_score = retrieved[0]
    if inject_hallucinated_citation:
        cited = next(c for c in CHUNKS if c['id'] not in [r['id'] for r, _ in retrieved])
        cited_id = cited['id']
    else:
        cited_id = top_chunk['id']
    latency_ms = (time.perf_counter() - t0) * 1000
    trace = {
        'question': question,
        'retrieved_ids': [c['id'] for c, _ in retrieved],
        'top_score': round(top_score, 3),
        'answer_text': top_chunk['text'],
        'cited_chunk_id': cited_id,
        'latency_ms': round(latency_ms, 3),
        'tokens_approx': len(top_chunk['text'].split()) + len(question.split()),  # word count as a token proxy
    }
    return trace

run_system(EVAL_SET[0]['question'])


### Two programmatic checks

**Faithfulness**: is the cited chunk actually one of the chunks retrieved? A system
that answers correctly but cites the wrong source is not a system a reader can
verify, and this check catches that failure mode for free, no model required. We
inject one hallucinated citation on purpose to prove the check actually catches
something, rather than trusting a check that has never seen a failure.

**Reference match**: does the reference phrase appear in the answer? This is
`exact match`, the crudest of the module's reference-based checks, and it is worth
seeing it fail honestly rather than only succeed.

In [ ]:
def check_faithfulness(trace):
    return trace['cited_chunk_id'] in trace['retrieved_ids']

def check_reference_match(trace, reference_answer):
    return reference_answer.lower() in trace['answer_text'].lower()

print(f"{'question':52s} {'faithful':>9s} {'ref_match':>10s}")
for row in EVAL_SET:
    t = run_system(row['question'])
    print(f"{row['question'][:52]:52s} {str(check_faithfulness(t)):>9s} "
          f"{str(check_reference_match(t, row['reference_answer'])):>10s}")

print()
print('same question, with an injected hallucinated citation:')
bad_trace = run_system(EVAL_SET[0]['question'], inject_hallucinated_citation=True)
print('cited:', bad_trace['cited_chunk_id'], '| retrieved:', bad_trace['retrieved_ids'],
      '| faithful:', check_faithfulness(bad_trace))


Notice the third question. `check_reference_match` fails it, even though the
system's answer is correct: the chunk says "1.5 times **the vessel's** maximum
allowable working pressure" and the reference answer says "1.5 times **the**
maximum allowable working pressure." Exact match is exactly this brittle, which is
the module's own point in naming it alongside embedding similarity and numeric
tolerance rather than presenting it as sufficient on its own.

## Part A continued: an LLM-as-judge, validated against hand labels

The rubric: does the answer, together with its citation, correctly and completely
answer the question, on a 1 (no) to 5 (yes, precisely) scale. Real practice asks a
strong model for a score and a justification in structured output; the stand-in
below scores lexical overlap with the reference answer only, which is honest about
being naive and, as the next cell shows, exposes a real blind spot rather than
hiding one.

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def content_words(text):
    return {w for w in re.findall(r"[a-z0-9']+", text.lower())
            if w not in ENGLISH_STOP_WORDS and len(w) > 1}

def heuristic_judge(answer_text, reference_answer):
    a, r = content_words(answer_text), content_words(reference_answer)
    if not r:
        return 3
    overlap = len(a & r) / len(r)
    if overlap >= 0.8: return 5
    if overlap >= 0.5: return 4
    if overlap >= 0.25: return 3
    if overlap > 0: return 2
    return 1


### The validation set: eight cases, hand-labeled

Four are the real eval-set questions, answered correctly. Four are constructed to
probe specific failure modes: a citation swapped for a wrong one (answer text
correct), a number swapped for a wrong one (citation correct), an answer to the
*wrong* question that happens to reuse similar words, and a correct answer worded
differently than the reference. The human scores are a considered judgment call
made once, in writing, the way A11 asks you to make yours.

In [ ]:
VALIDATION_CASES = [
    ('q1-correct',
     "For 12 mm outer diameter soft copper tube, the minimum bend radius using a manual lever-type bender is 48 mm.",
     '48 mm', 5, 'correct value, correctly cited'),
    ('q2-correct',
     "Vessels in continuous service must receive an internal visual inspection no less often than every five years.",
     'every five years', 5, 'correct value, correctly cited'),
    ('q3-wrong-question',
     "The tube must not be bent tighter than four tube-diameters' worth of radius when working it by hand.",
     '48 mm', 2, 'answers the GENERAL rule, not the 12mm-specific value asked for, despite reusing similar words'),
    ('q4-hallucinated-citation',
     "For 12 mm outer diameter soft copper tube, the minimum bend radius using a manual lever-type bender is 48 mm.",
     '48 mm', 1, 'right text, but the citation points at a different chunk -- unverifiable, not a minor issue'),
    ('q5-hallucinated-citation-2',
     "Dry, unlubricated Grade 5 bolt torque values include 30 ft-lb for a 3/8 inch bolt.",
     '30 ft-lb', 1, 'same failure as q4: right text, wrong citation'),
    ('q6-wrong-number',
     "For 12 mm outer diameter soft copper tube, the minimum bend radius using a manual lever-type bender is 36 mm.",
     '48 mm', 1, 'wrong number, correctly cited to the right clause -- the citation makes this MORE dangerous, not less'),
    ('q7-correct-reworded',
     "A 3/8-inch fastener of this grade should be torqued to thirty foot-pounds.",
     '30 ft-lb', 4, 'correct and well-cited, just phrased differently than the reference string'),
    ('q8-vague',
     "The hydrostatic test pressure is related to the vessel's maximum allowable working pressure.",
     '1.5 times the maximum allowable working pressure', 2, "drops the 1.5x multiplier a maintenance engineer actually needs"),
]

print(f"{'case':26s} {'judge':>5s} {'human':>5s}  reason")
judge_scores, human_scores = [], []
for case_id, answer, ref, human, reason in VALIDATION_CASES:
    j = heuristic_judge(answer, ref)
    judge_scores.append(j); human_scores.append(human)
    flag = '  <-- disagree' if abs(j - human) >= 2 else ''
    print(f"{case_id:26s} {j:5d} {human:5d}  {reason}{flag}")


### Agreement, measured, not assumed

Bucket both scales into low/mid/high (the module's own suggestion for keeping this
simple) and compute both the raw agreement rate and Cohen's kappa, which corrects
for the agreement you would expect from chance alone.

In [ ]:
from sklearn.metrics import cohen_kappa_score

def bucket(s):
    return 'low' if s <= 2 else ('mid' if s == 3 else 'high')

j_buckets = [bucket(s) for s in judge_scores]
h_buckets = [bucket(s) for s in human_scores]
agreement = sum(a == b for a, b in zip(j_buckets, h_buckets)) / len(VALIDATION_CASES)
kappa = cohen_kappa_score(j_buckets, h_buckets)

print(f'bucketed agreement: {agreement:.2f}')
print(f"Cohen's kappa: {kappa:.2f}")


Read that kappa for what it is: at or below zero means this judge agrees with the
human labels no better than chance would, on this validation set. Catching a result
like that before trusting the judge on the other hundred questions you did not have
time to hand-label is exactly why this check comes first. A judge failing this badly
is exactly what validation exists to catch. Every disagreement above traces to a
specific, nameable blind spot: this heuristic cannot tell a correct citation from a
hallucinated one, and it cannot tell a right number from a wrong one sitting in an
otherwise identical sentence. A real LLM judge, asked the right rubric question,
would very likely close the q7 paraphrase gap; whether it closes the
citation-blindness gap depends entirely on whether your rubric explicitly asked it to
check the citation: the rubric's wording determines the outcome here more than the
model behind it does.

## Logging both parts to MLflow

One run per eval pass, aggregate metrics plus the full trace as an artifact, using
a local file-backed tracking store so this runs with no server, consistent with
L2's MLflow setup.

In [ ]:
import mlflow
import json
from pathlib import Path

mlflow.set_tracking_uri('sqlite:///l21_mlflow.db')
mlflow.set_experiment('l21-rag-eval')

traces = [run_system(row['question']) for row in EVAL_SET]
faithfulness_rate = np.mean([check_faithfulness(t) for t in traces])
ref_match_rate = np.mean([check_reference_match(t, row['reference_answer'])
                           for t, row in zip(traces, EVAL_SET)])
mean_latency_ms = np.mean([t['latency_ms'] for t in traces])

with mlflow.start_run(run_name='rag-eval-pass'):
    mlflow.log_metric('faithfulness_rate', faithfulness_rate)
    mlflow.log_metric('reference_match_rate', ref_match_rate)
    mlflow.log_metric('judge_human_kappa', kappa)
    mlflow.log_metric('mean_latency_ms', mean_latency_ms)
    Path('traces.jsonl').write_text('\n'.join(json.dumps(t) for t in traces))
    mlflow.log_artifact('traces.jsonl')

print(f'faithfulness_rate={faithfulness_rate:.2f}  reference_match_rate={ref_match_rate:.2f}  '
      f'mean_latency_ms={mean_latency_ms:.3f}')


## Part B: evaluating a regression model by slice, on real data

Same Intel Lab Parquet file as L3, L4, and L19. The task: predict humidity from
temperature, light, and voltage. This section is about what a single aggregate metric
hides about a model's behavior, using this regression model as the example.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)
PARQUET = CACHE / 'readings.parquet'
URL = 'https://raw.githubusercontent.com/linsea423/Intel_Lab_Data/master/data.zip'
COLS = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']

if not PARQUET.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        with z.open('data.txt') as f:
            raw = pd.read_csv(f, sep=r'\s+', header=None, names=COLS, on_bad_lines='skip')
    df = raw.dropna(subset=['moteid']).copy()
    df['moteid'] = pd.to_numeric(df['moteid'], errors='coerce')
    df = df.dropna(subset=['moteid'])
    df['moteid'] = df['moteid'].astype(int)
    df = df[df.moteid.between(1, 54)]
    df['ts'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='mixed', errors='coerce')
    df = df.dropna(subset=['ts'])
    for c in ['temperature', 'humidity', 'light', 'voltage']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['temperature', 'humidity', 'light', 'voltage'])
    df[['moteid', 'ts', 'temperature', 'humidity', 'light', 'voltage']].sort_values(
        ['moteid', 'ts']).to_parquet(PARQUET, index=False)

df = pd.read_parquet(PARQUET)
df = df[(df.temperature.between(0, 50)) & (df.humidity.between(0, 100)) & (df.voltage.between(1.5, 3.0))]
print(df.shape, 'rows after basic range cleaning')


### Fit on the trustworthy slice, then evaluate on both slices

L3 established 2.4 V as the threshold below which a mote's readings become
suspect. Train and validate only on readings at or above that threshold, then see
what the same model does on the readings below it, which it has never seen and was
never meant to be trusted on.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURES = ['temperature', 'light', 'voltage']
TARGET = 'humidity'

good = df[df.voltage >= 2.4]
low_voltage = df[df.voltage < 2.4]

X_train, X_test, y_train, y_test = train_test_split(
    good[FEATURES], good[TARGET], test_size=0.2, random_state=0)

model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0, n_jobs=-1)
model.fit(X_train.to_numpy(), y_train.to_numpy())

pred_good = model.predict(X_test.to_numpy())
pred_low = model.predict(low_voltage[FEATURES].to_numpy())

mae_good = mean_absolute_error(y_test, pred_good)
mae_low = mean_absolute_error(low_voltage[TARGET], pred_low)
print(f'held-out, GOOD-voltage slice:  MAE = {mae_good:.2f}  (n={len(y_test)})')
print(f'LOW-voltage slice:             MAE = {mae_low:.2f}  (n={len(low_voltage)})')
print(f'error ratio: {mae_low / mae_good:.1f}x worse on the low-voltage slice')


A single "MAE = 2.5" reported for the whole dataset would have buried this
entirely: the low-voltage slice is a real, substantial fraction of the data, and
the model is measurably worse on it, in exactly the region L3 already told you not
to trust. This is the module's "aggregate metrics lie" pitfall, shown rather than
asserted, on the same dataset the rest of this course has been building on.

### A bootstrap confidence interval, because one number is still one number

Even the good-slice MAE above is a point estimate from one held-out split.
Bootstrap resampling the residuals gives an interval around it.

In [ ]:
rng = np.random.default_rng(0)
resid = y_test.to_numpy() - pred_good
boot_rmse = [np.sqrt(np.mean(resid[rng.integers(0, len(resid), len(resid))] ** 2)) for _ in range(2000)]
lo, hi = np.percentile(boot_rmse, [2.5, 97.5])
print(f'good-slice RMSE: {np.sqrt(np.mean(resid**2)):.3f}, 95% bootstrap CI [{lo:.3f}, {hi:.3f}]')


### Calibration: does the model's own uncertainty mean anything?

A random forest's per-tree spread is a common, convenient stand-in for a
predictive interval. It is worth checking whether it is *any good* before you
report it to anyone, exactly as L13 argued for a Gaussian process's interval.

In [ ]:
def empirical_coverage(model, X, y_true):
    tree_preds = np.stack([t.predict(X) for t in model.estimators_], axis=0)
    lo_q, hi_q = np.percentile(tree_preds, [2.5, 97.5], axis=0)
    return np.mean((y_true >= lo_q) & (y_true <= hi_q))

cov_good = empirical_coverage(model, X_test.to_numpy(), y_test.to_numpy())
cov_low = empirical_coverage(model, low_voltage[FEATURES].to_numpy(), low_voltage[TARGET].to_numpy())
print(f'nominal interval: 95%')
print(f'empirical coverage, good-voltage slice: {cov_good:.1%}')
print(f'empirical coverage, low-voltage slice:  {cov_low:.1%}')


Read those two numbers slowly. A "95% interval" that actually contains the true value
well under a fifth of the time is failing at the one job an interval has to do. The
forest's tree-to-tree spread captures a sliver of model uncertainty and none of the
residual noise a real predictive interval has to cover, and nothing about training
the model warns you of this; only checking coverage against held-out truth does. Log
this next to the MAE numbers, not instead of them.

In [ ]:
with mlflow.start_run(run_name='surrogate-eval-pass'):
    mlflow.log_metric('mae_good_slice', mae_good)
    mlflow.log_metric('mae_low_voltage_slice', mae_low)
    mlflow.log_metric('rmse_good_slice_ci_lo', lo)
    mlflow.log_metric('rmse_good_slice_ci_hi', hi)
    mlflow.log_metric('coverage_good_slice', cov_good)
    mlflow.log_metric('coverage_low_voltage_slice', cov_low)

print('logged to sqlite:///l21_mlflow.db, experiment \'l21-rag-eval\'')


Both eval passes now live in the same MLflow store: aggregate and per-slice ML
metrics, and the RAG harness's faithfulness rate, reference-match rate, and
judge/human kappa, all queryable and comparable the next time either system changes.
That comparability across runs, not any single number this notebook prints, is the
actual deliverable of an eval harness.

Full notes, with the ML-metrics vocabulary and the observability checklist this
notebook only exercises: [`../notes.md`](notes.md).